# WikiLens · Qwen prompt comparison

Three generation configurations over **the same seven cross-encoder-selected passages**. No search, embedding or reranking is repeated. Existing notebooks are preserved.

1. Current prompt, thinking off, greedy decoding.
2. Research-editor prompt, thinking off, greedy decoding.
3. Research-editor prompt, native thinking on, sampled decoding.

The third configuration changes both thinking and decoding. This is a practical configuration comparison, not an isolated causal test of CoT. All use the same pinned Qwen3-1.7B weights, 500-word target, 8192-token context and 4000-token output budget. Reasoning consumes part of that output budget. Only final article text is displayed; raw diagnostics are retained separately.

Completed results and assistant historical ratings are displayed in the final cell. Detailed claim checks distinguish incorrect facts, imprecision and unsupported interpretations; citation compliance is a separate measure.


## Settings

Keep these fixed across variants for comparison. Input passages and question come from the frozen report.


In [ ]:
SETTINGS = {
    "source_report": "reports/reranker-comparison-20261114T104507Z.json",
    "answer_language": "English",
    "target_words": 500,
    "model_context_tokens": 8192,
    "max_new_tokens": 4000,  # shared ceiling: reasoning + final article
    "max_time": 300,
    "seed": 42,
}


## 1 · Load the frozen evidence and prompts

Use the project GPU kernel. This cell only reads saved results; it loads no model weights.


In [ ]:
from pathlib import Path
import sys, json, html, hashlib, importlib
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display, HTML, Markdown
project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
assert sys.version_info[:2] == (3, 12), "Select the project Python 3.12 kernel."
assert Path(sys.prefix).resolve() == (project_dir / ".venv").resolve(), "Select the project .venv kernel."
import wikilens.prompt_comparison as prompt_module
prompt_module = importlib.reload(prompt_module)
source_path = project_dir / SETTINGS["source_report"]
capture = json.loads(source_path.read_text())
shared_run = capture["variants"]["cross_encoder"]["run"]
shared_nodes = shared_run["stages"]["postprocessing"]["nodes"]
results = {}
print("Question:", shared_run["question"], "| focus:", shared_run["search_focus"])
print("Pinned Qwen:", shared_run["runtime"]["revisions"]["qwen"])
print("Shared evidence:", len(shared_nodes), "passages,", sum(len(n["text"]) for n in shared_nodes), "characters")
display(pd.DataFrame([{"id":n["id"], "title":n["title"], "lang":n["lang"], "section":" > ".join(n.get("heading_path",[]))} for n in shared_nodes]))
for name, spec in prompt_module.VARIANTS.items():
    system = (shared_run["stages"]["generation"]["messages"][0]["content"]
        if spec["prompt"] == "current" else prompt_module.EDITOR_PROMPT.format(**SETTINGS))
    display(HTML("<details><summary>" + html.escape(name) + " prompt and decoding</summary><pre>"
        + html.escape(json.dumps(spec, indent=2) + "\n\n" + system) + "</pre></details>"))

def show_result(result):
    print("Status:", result["status"], "| time:", round(result["seconds"], 2), "s",
          "| generated tokens:", result.get("generated_tokens"), "| reasoning tokens:", result.get("reasoning_tokens"))
    if result.get("final_text"):
        display(Markdown("*AI draft — historical accuracy not yet established.*\n\n" + html.escape(result["final_text"])))
    else:
        print(result.get("error", "No final article was produced before the output budget ended."))
    display(HTML("<details><summary>Raw diagnostic output</summary><pre>"
        + html.escape(result.get("raw_output", "")) + "</pre></details>"))


## 2 · Current prompt

One real generation call. This cell releases its owned GPU model afterwards.


In [ ]:
results["current"] = prompt_module.run_variant(capture, "current", SETTINGS)
show_result(results["current"])


## 3 · Editor prompt

One real generation call. This cell releases its owned GPU model afterwards.


In [ ]:
results["editor"] = prompt_module.run_variant(capture, "editor", SETTINGS)
show_result(results["editor"])


## 4 · Editor prompt + native thinking

One real generation call. This cell releases its owned GPU model afterwards.


In [ ]:
results["editor_thinking"] = prompt_module.run_variant(capture, "editor_thinking", SETTINGS)
show_result(results["editor_thinking"])


## 5 · Compare and save

Generation timings include imports, model loading and cleanup. Citation markers are counted, not treated as proof of support. Historical scores are recorded separately after reviewing the actual outputs.


In [ ]:
summary = []
for name, result in results.items():
    import re
    summary.append({"variant":name, "generation_s":round(result["seconds"],2),
        "words":len(result.get("final_text", "").split()), "reasoning_tokens":result.get("reasoning_tokens"),
        "generated_tokens":result.get("generated_tokens"), "finish":result.get("finish_reason"),
        "citation_markers":len(re.findall(r"\[S\d+\]",result.get("final_text", "")))})
display(pd.DataFrame(summary))
report = {"created_at":datetime.now(timezone.utc).isoformat(), "settings":SETTINGS,
    "source_sha256":hashlib.sha256(source_path.read_bytes()).hexdigest(),
    "question":shared_run["question"], "nodes":shared_nodes,
    "review_rubric":prompt_module.REVIEW_RUBRIC, "variants":results,
    "limitation":"Single fixed evidence set. Thinking configuration also changes decoding; no causal CoT claim."}
report_path = project_dir / "reports" / ("prompt-comparison-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".json")
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n")
print("Saved:",report_path)


## Historical review and saved outputs

The last cell reads the completed comparison and the assistant’s subsequent claim-by-claim historical review. Scores concern historical accuracy; citation compliance is separate. No new model calls.


In [ ]:
from pathlib import Path
import json, html
import pandas as pd
from IPython.display import display, Markdown, HTML
project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
saved_path = project_dir / "reports/prompt-comparison-latest.json"
if not saved_path.exists():
    print("Run scripts/compare-prompts.py first.")
else:
    saved = json.loads(saved_path.read_text())
    display(pd.DataFrame([{"variant":name,"seconds":round(item["seconds"],2),
        "words":len(item.get("final_text", "").split()),
        "historical_accuracy_10":(item.get("historical_review") or {}).get("score"),
        "citation_markers":item.get("citation_markers")}
        for name,item in saved["variants"].items()]))
    for name,item in saved["variants"].items():
        display(Markdown("### " + name + "\n\n" + html.escape(item.get("final_text") or item.get("error", "No final answer"))))
        if item.get("historical_review"):
            display(Markdown("**Assistant historical review:**\n\n" + item["historical_review"]["summary"]))

            checks = []
            for check in item["historical_review"].get("claims", []):
                links = " ".join('<a href="' + html.escape(source["url"], quote=True) + '" target="_blank" rel="noopener">' + html.escape(source["title"]) + '</a>' for source in check.get("sources", []))
                checks.append("<li><strong>" + html.escape(check["verdict"]) + "</strong> · " + html.escape(check["claim"]) + "<p>" + html.escape(check["explanation"]) + " " + links + "</p></li>")
            display(HTML("<details><summary>Historical claim checks and sources</summary><ol>" + "".join(checks) + "</ol></details>"))
